# Integrating PyFluent with PyWorkbench:
This example showcases how to use PyFluent workflow together with PyWorkbench - (Python client scripting for Ansys Workbench).

This example sets up and solves a three-dimensional turbulent fluid flow
and heat transfer problem in a mixing elbow, which is common in piping
systems in power plants and process industries. Predicting the flow field
and temperature field in the area of the mixing region is important to
designing the junction properly.

This example uses Pyfluent settings objects API's.

## Problem description

A cold fluid at 20 deg C flows into the pipe through a large inlet. It then mixes
with a warmer fluid at 40 deg C that enters through a smaller inlet located at
the elbow. The pipe dimensions are in inches, and the fluid properties and
boundary conditions are given in SI units. Because the Reynolds number for the
flow at the larger inlet is ``50,800``, a turbulent flow model is required.

## Performed required imports
Performing essential imports for Ansys Workbench, Fluent Pythonic Interface and for downloading examples data.

In [1]:
import pathlib
from ansys.workbench.core import launch_workbench
import ansys.fluent.core as pyfluent
from ansys.fluent.core import examples

## Specify client and server directories with launch WB service.

In [2]:
workdir = pathlib.Path("__file__").parent

In [3]:
wb = launch_workbench(client_workdir=str(workdir.absolute()), show_gui=False, version="251")

## Download the input file from example data and upload to server directory.

In [4]:
import_filename = examples.download_file("mixing_elbow.msh.h5", "pyfluent/mixing_elbow")
wb.upload_file(import_filename)

Uploading mixing_elbow.msh.h5:   0%|          | 0.00/2.68M [00:00<?, ?B/s]

Uploading mixing_elbow.msh.h5: 100%|██████████| 2.68M/2.68M [00:00<00:00, 84.8MB/s]

## Generate a "FLUENT" System using Ansys Workbench Scripting API (used for Journaling) and parse it to the PyWorkbench API.

In [5]:
export_path = "wb_log_file.log"
wb.set_log_file(export_path)
wb.run_script_string('template1 = GetTemplate(TemplateName="FLUENT")', log_level="info")
wb.run_script_string("system1 = template1.CreateSystem()")

{}

## Launch Fluent & Connect to Fluent
Launch Fluent as server with PyWorkbench API and and connect to Pyfluent session

In [6]:
server_info_file = wb.start_fluent_server(system_name="FLU")
fluent_session = pyfluent.connect_to_fluent(server_info_file_name=server_info_file)

## Import mesh and perform mesh check

In [7]:
# Import the mesh and perform a mesh check, which lists the minimum and maximum
# x, y, and z values from the mesh in the default SI units of meters. The mesh
# check also reports a number of other mesh features that are checked. Any errors
# in the mesh are reported. Ensure that the minimum volume is not negative because
# Fluent cannot begin a calculation when this is the case.

fluent_session.file.read_mesh(file_name=import_filename)
fluent_session.mesh.check()


pyfluent.settings_api WARNING: Mismatch between generated file and server object info. Dynamically created settings classes will be used.


'file' is deprecated. Use 'settings.file' instead.
Fast-loading "C:\PROGRA~1\ANSYSI~1\v251\fluent\fluent25.1.0\\addons\afd\lib\hdfio.bin"
Done.


Multicore SMT processors detected. Processor affinity set!



Reading from pyworkbench:"C:\Users\ansys\actions-runner\_work\pyworkbench-examples\pyworkbench-examples\pyworkbench-examples\doc\source\examples\pyfluent-workflow\mixing_elbow.msh.h5" in NODE0 mode ...
  Reading mesh ...
       17822 cells,     1 cell zone  ...
          17822 mixed cells,  zone id: 87
       91581 faces,     7 face zones ...
           2168 polygonal wall faces,  zone id: 34
            268 polygonal wall faces,  zone id: 33
            155 polygonal pressure-outlet faces,  zone id: 32
            152 polygonal velocity-inlet faces,  zone id: 31
             55 polygonal velocity-inlet faces,  zone id: 30
           2001 polygonal symmetry faces,  zone id: 29
          86782 mixed interior faces,  zone id: 89
       66417 nodes,     3 node zones ...



Building...


...

Done.



Writing Settings file "C:\Users\ansys\AppData\Local\Temp\WB_ansys_7448_2\wbnew_files\dp0\FLU\Fluent\mixing_elbow.set"...


	writing rp variables ... Done.
	writing domain variables ... Done.
	writing elbow-fluid (type fluid) (mixture) ... Done.
	writing interior--elbow-fluid (type interior) (mixture) ... Done.
	writing symmetry-xyplane (type symmetry) (mixture) ... Done.
	writing hot-inlet (type velocity-inlet) (mixture) ... Done.
	writing cold-inlet (type velocity-inlet) (mixture) ... Done.
	writing outlet (type pressure-outlet) (mixture) ... Done.
	writing wall-inlet (type wall) (mixture) ... Done.
	writing wall-elbow (type wall) (mixture) ... Done.
	writing zones map name-id ... Done.


'mesh' is deprecated. Use 'settings.mesh' instead.

 Domain Extents:
   x-coordinate: min (m) = -2.000000e-01, max (m) = 2.000000e-01
   y-coordinate: min (m) = -2.250000e-01, max (m) = 2.000000e-01
   z-coordinate: min (m) = 0.000000e+00, max (m) = 4.992265e-02
 Volume statistics:
   minimum volume (m3): 2.443845e-10
   maximum volume (m3): 5.720378e-07
     total volume (m3): 2.500656e-03
 Face area statistics:
   minimum face area (m2): 3.226186e-08
   maximum face area (m2): 7.873458e-05


 Checking mesh............................
Done.

Note: Settings to improve the robustness of pathline and
      particle tracking have been automatically enabled.



## Set working units for mesh

Set the working units for the mesh to inches. Because the default SI units are
used for everything except length, you do not have to change any other units
in this example. If you want working units for length to be other than inches
(for example, millimeters), make the appropriate change.

In [8]:
fluent_session.tui.define.units("length", "in")

The following solver settings object method could also be used to execute the above command:
<solver_session>.settings.setup.general.units.set_units(quantity = "length", units_name = "in", scale_factor = 1., offset = 0.)


## Enable heat transfer
Enable heat transfer by activating the energy equation.

In [9]:
fluent_session.setup.models.energy.enabled = True

'setup' is deprecated. Use 'settings.setup' instead.


## Create material
Create a material named ``"water-liquid"``.

In [10]:
fluent_session.setup.materials.database.copy_by_name(type="fluid", name="water-liquid")

 
Material "water-liquid" copied from database: fluent-database.



## Set up cell zone conditions

Set up the cell zone conditions for the fluid zone (``elbow-fluid``). Set ``material``
to ``"water-liquid"``.

In [11]:
fluent_session.setup.cell_zone_conditions.fluid['elbow-fluid'].general.material = "water-liquid"

## Set up boundary conditions for CFD analysis

Set up the boundary conditions for the inlets, outlet, and walls for CFD
analysis.
- cold inlet (cold-inlet), Setting: Value:
- Velocity Specification Method: Magnitude, Normal to Boundary
- Velocity Magnitude: 0.4 [m/s]
- Specification Method: Intensity and Hydraulic Diameter
- Turbulent Intensity: 5 [%]
- Hydraulic Diameter: 4 [inch]
- Temperature: 293.15 [K]

In [12]:
cold_inlet = fluent_session.setup.boundary_conditions.velocity_inlet["cold-inlet"]
cold_inlet.get_state()
cold_inlet.momentum.velocity.value = 0.4
cold_inlet.turbulence.turbulent_specification = "Intensity and Hydraulic Diameter"
cold_inlet.turbulence.turbulent_intensity = 0.05
cold_inlet.turbulence.hydraulic_diameter = "4 [in]"
cold_inlet.thermal.t.value = 293.15

Note: A newer syntax is available to perform the last operation:
solver.settings.setup.boundary_conditions.velocity_inlet['cold-inlet'].momentum.velocity_magnitude.value = 0.4

Execute the following code to suppress future warnings like the above:

>>> import warnings
>>> warnings.filterwarnings("ignore", category=DeprecatedSettingWarning)
Note: A newer syntax is available to perform the last operation:
solver.settings.setup.boundary_conditions.velocity_inlet['cold-inlet'].turbulence.turbulence_specification = "Intensity and Hydraulic Diameter"


Note: A newer syntax is available to perform the last operation:
solver.settings.setup.boundary_conditions.velocity_inlet['cold-inlet'].thermal.temperature.value = 293.15


- hot inlet (hot-inlet), Setting: Value:
- Velocity Specification Method: Magnitude, Normal to Boundary
- Velocity Magnitude: 1.2 [m/s]
- Specification Method: Intensity and Hydraulic Diameter
- Turbulent Intensity: 5 [%]
- Hydraulic Diameter: 1 [inch]
- Temperature: 313.15 [K]

In [13]:
hot_inlet = fluent_session.setup.boundary_conditions.velocity_inlet["hot-inlet"]
hot_inlet.momentum.velocity.value = 1.2
hot_inlet.turbulence.turbulent_specification = "Intensity and Hydraulic Diameter"
hot_inlet.turbulence.turbulent_intensity = 0.05
hot_inlet.turbulence.hydraulic_diameter = "1 [in]"
hot_inlet.thermal.t.value = 313.15

Note: A newer syntax is available to perform the last operation:
solver.settings.setup.boundary_conditions.velocity_inlet['hot-inlet'].momentum.velocity_magnitude.value = 1.2


Note: A newer syntax is available to perform the last operation:
solver.settings.setup.boundary_conditions.velocity_inlet['hot-inlet'].turbulence.turbulence_specification = "Intensity and Hydraulic Diameter"


Note: A newer syntax is available to perform the last operation:
solver.settings.setup.boundary_conditions.velocity_inlet['hot-inlet'].thermal.temperature.value = 313.15


- pressure outlet (outlet), Setting: Value:
- Backflow Turbulent Intensity: 5 [%]
- Backflow Turbulent Viscosity Ratio: 4

In [14]:
fluent_session.setup.boundary_conditions.pressure_outlet[
    "outlet"
].turbulence.backflow_turbulent_viscosity_ratio = 4

## Initialize flow field

In [15]:
fluent_session.solution.initialization.hybrid_initialize()

'solution' is deprecated. Use 'settings.solution' instead.



Initialize using the hybrid initialization method.

Checking case topology... 
-This case has both inlets & outlets 
-Pressure information is not available at the boundaries.
 Case will be initialized with constant pressure



	iter		scalar-0

	1		1.000000e+00


	2		1.763140e-04
	3		1.043067e-05
	4		9.566706e-06


	5		1.769730e-06
	6		2.165014e-06


	7		3.173055e-07
	8		5.159712e-07


	9		1.080573e-07


	10		1.128050e-07

Hybrid initialization is done.


## Solve for 150 iterations
Setting iteration count to 150 to solve the model.

In [16]:
fluent_session.solution.run_calculation.iter_count = 100

## Update Solution using Workbench Journal Commands

In [17]:
script_string = """
solutionComponent1 = system1.GetComponent(Name="Solution")
system1 = GetSystem(Name="FLU")
solutionComponent1 = system1.GetComponent(Name="Solution")
solutionComponent1.Update(AllDependencies=True)
"""

In [18]:
wb.run_script_string(script_string)


Writing to pyworkbench:"C:\Users\ansys\AppData\Local\Temp\WB_ansys_7448_2\wbnew_files\dp0\FLU\Fluent\mixing_elbow-Setup-Output.cas.h5" in NODE0 mode and compression level 1 ...
Grouping cells for Laplace smoothing ...
         Combining every 2 partitions.
  Done.       17822 cells,     1 zone  ...
       91581 faces,     7 zones ...
       66417 nodes,     1 zone  ...
  Done.


Done.

Writing Settings file "C:\Users\ansys\AppData\Local\Temp\WB_ansys_7448_2\wbnew_files\dp0\FLU\Fluent\mixing_elbow.set"...


	writing rp variables ... Done.
	writing domain variables ... Done.


	writing elbow-fluid (type fluid) (mixture) ... Done.
	writing interior--elbow-fluid (type interior) (mixture) ... Done.
	writing symmetry-xyplane (type symmetry) (mixture) ... Done.
	writing hot-inlet (type velocity-inlet) (mixture) ... Done.
	writing cold-inlet (type velocity-inlet) (mixture) ... Done.
	writing outlet (type pressure-outlet) (mixture) ... Done.
	writing wall-inlet (type wall) (mixture) ... Done.
	writing wall-elbow (type wall) (mixture) ... Done.
	writing zones map name-id ... Done.



Writing to pyworkbench:"C:\Users\ansys\AppData\Local\Temp\WB_ansys_7448_2\wbnew_files\dp0\FLU\Fluent\mixing_elbow-1.cas.h5" in NODE0 mode and compression level 1 ...
Grouping cells for Laplace smoothing ...
       17822 cells,     1 zone  ...
       91581 faces,     7 zones ...
       66417 nodes,     1 zone  ...
  Done.


...


Writing to pyworkbench:"C:\Users\ansys\AppData\Local\Temp\WB_ansys_7448_2\wbnew_files\dp0\FLU\Fluent\mixing_elbow-1.cas.h5" in NODE0 mode and compression level 1 ...
Grouping cells for Laplace smoothing ...
       17822 cells,     1 zone  ...
       91581 faces,     7 zones ...
       66417 nodes,     1 zone  ...
  Done.


Done.



Writing to pyworkbench:"C:\Users\ansys\AppData\Local\Temp\WB_ansys_7448_2\wbnew_files\dp0\FLU\Fluent\mixing_elbow-1-00094.dat.h5" in NODE0 mode and compression level 1 ...
  Writing results.


Done.


{}

## Postprocessing
Create and display velocity vectors on the ``symmetry-xyplane`` plane.

## Configure graphics picture export
Since Fluent is being run without the GUI, you must to export plots as picture files. Edit the picture settings to use a custom resolution so that the images are large enough.

In [19]:
graphics = fluent_session.results.graphics
if graphics.picture.use_window_resolution.is_active():
    graphics.picture.use_window_resolution = False
graphics.picture.x_resolution = 1920
graphics.picture.y_resolution = 1440

'results' is deprecated. Use 'settings.results' instead.


## Create velocity vectors
Create and display velocity vectors on the ``symmetry-xyplane`` plane. Then, export the image for inspection.

In [20]:
graphics = fluent_session.results.graphics

In [21]:
graphics.vector["velocity_vector_symmetry"] = {}
velocity_symmetry = fluent_session.results.graphics.vector["velocity_vector_symmetry"]
velocity_symmetry.print_state()
velocity_symmetry.field = "velocity-magnitude"
velocity_symmetry.surfaces_list = [
    "symmetry-xyplane",
]
velocity_symmetry.scale.scale_f = 4
velocity_symmetry.style = "arrow"
velocity_symmetry.display()


name : velocity_vector_symmetry
vector_field : velocity
field : velocity-magnitude
range_option : 
  option : auto-range-on
  auto_range_on : 
    global_range : True
range_options : 
  global_range : True
  auto_range : True
  clip_to_range : False
  minimum : 0.0
  maximum : 0.0
options : 
  auto_scale : True
  vector_style : 3d arrow
  scale : 1
  skip : 0
scale : 
  auto_scale : True
  scale_f : 1
style : 3d arrow
skip : 0
vector_opt : 
  in_plane : False
  fixed_length : False
  x_comp : True
  y_comp : True
  z_comp : True
  scale_head : 0.3
  tessellation : 24
  color : 
color_map : 
  visible : True
  color : field-velocity
  size : 100
  log_scale : False
  format : %0.2e
  user_skip : 9
  show_all : True
  position : 1
  font_name : Helvetica
  font_automatic : True
  font_size : 0.032
  length : 0.54
  width : 6.0
  bground_transparent : True
  bground_color : #CCD3E2
  title_elements : Variable and Object Name
display_state_name : None


In [22]:
graphics.views.restore_view(view_name="front")
graphics.views.auto_scale()
graphics.picture.save_picture(file_name="velocity_vector_symmetry.png")

## Compute mass flow rate
Compute the mass flow rate.

In [23]:
fluent_session.solution.report_definitions.flux["mass_flow_rate"] = {}

mass_flow_rate = fluent_session.solution.report_definitions.flux["mass_flow_rate"]
mass_flow_rate.boundaries = [
    "cold-inlet",
    "hot-inlet",
    "outlet",
]
mass_flow_rate.print_state()
fluent_session.solution.report_definitions.compute(report_defs=["mass_flow_rate"])

'solution' is deprecated. Use 'settings.solution' instead.



name : mass_flow_rate
report_type : flux-massflow
boundaries : 
  0 : cold-inlet
  1 : hot-inlet
  2 : outlet
per_zone : False
average_over : 1
retain_instantaneous_values : False
output_parameter : False

      mass_flow_rate
 -------------------
                  Mass Flow Rate              [kg/s]


[{'mass_flow_rate': [-5.185604095458984e-06, 0]}]

-------------------------------- -------------------
                  mass_flow_rate      -5.1856041e-06


## Exit Fluent Session

In [24]:
fluent_session.exit()

## Save project

In [25]:
save_string = """import os
workdir = GetServerWorkingDirectory()
path = os.path.join(workdir, "mixing_elbow.wbpj")
Save(FilePath=path , Overwrite=True)"""
wb.run_script_string(save_string)

{}

## Archive Project

In [26]:
archive_string ="""import os
workdir = GetServerWorkingDirectory()
path = os.path.join(workdir, "mixing_elbow.wbpz")
Archive(FilePath=path , IncludeExternalImportedFiles=True)"""
wb.run_script_string(archive_string)

{}

## Download the archived project which has all simulation data and results.

In [27]:
wb.download_file("mixing_elbow.wbpz")

'mixing_elbow.wbpz'

## Exit Workbench Session.

In [28]:
wb.exit()